# Generation G - S1E4 - Tuning

This notebook is the companion of posts about Generative AI.

This episode shows how to use ChatCompletion API

## Conclusion 

requires a chain to feed the prompt into the chat

Resources

Other Chat APIs: https://api.python.langchain.com/en/latest/modules/chat_models.html

# Material

## Initializations

In [2]:
### Update environment

In [3]:
!apt-get update && apt-get install -y build-essential 1>/dev/null

Get:1 http://deb.debian.org/debian bullseye InRelease [116 kB]
Get:2 http://security.debian.org/debian-security bullseye-security InRelease [48.4 kB]
Get:3 http://deb.debian.org/debian bullseye-updates InRelease [44.1 kB]
Get:4 http://security.debian.org/debian-security bullseye-security/main amd64 Packages [251 kB]
Get:5 http://deb.debian.org/debian bullseye/main amd64 Packages [8183 kB]
Get:6 http://deb.debian.org/debian bullseye-updates/main amd64 Packages [17.3 kB]
Fetched 8660 kB in 2s (5366 kB/s)
Reading package lists... Done
debconf: delaying package configuration, since apt-utils is not installed


In [4]:
!apt-get update && apt-get install -y jq 1>/dev/null

Hit:1 http://deb.debian.org/debian bullseye InRelease
Hit:2 http://deb.debian.org/debian bullseye-updates InRelease
Hit:3 http://security.debian.org/debian-security bullseye-security InRelease
Reading package lists... Done
debconf: delaying package configuration, since apt-utils is not installed


In [5]:
!pip install --upgrade pip  1>/dev/null

## Requirements

In [6]:
!pip install langchain==0.0.230 1>/dev/null

In [7]:
!pip install openai==0.27.8 1>/dev/null

In [8]:
!pip install tiktoken==0.4.0 1>/dev/null

## Secrets and credentials

In [9]:
%%bash --out secrets 
# using AWS's Secret Manager to store keys
# garb the keys and store it into a Pytthon variable
export RESPONSE=$(aws secretsmanager get-secret-value --secret-id 'salvia/labbench/tests' )
export SECRETS=$( echo $RESPONSE | jq '.SecretString | fromjson')

echo $SECRETS

In [10]:
import os

os.environ["OPENAI_API_KEY"] = eval(secrets)["OPENAI_API_KEY"]


## Code session

```python
from langchain.chat_models import ChatOpenAI
chatopenai = ChatOpenAI(model_name="gpt-3.5-turbo")
query = "What is the distance to the Moon?"
response = chatopenai(query)
print(response)
````

result in NotImplementedError: OpenAICallbackHandler does not implement `on_chat_model_start`


In [19]:
%%time
from langchain.chat_models import ChatOpenAI
chatopenai = ChatOpenAI(model_name="gpt-3.5-turbo")
query = "What is the distance to the Moon?"
response = chatopenai.predict(query)
print(response)


The average distance from the Earth to the Moon is approximately 238,900 miles (384,400 kilometers).
CPU times: user 21.3 ms, sys: 0 ns, total: 21.3 ms
Wall time: 1.07 s


## chat completion model

In [21]:
%%time
from langchain.chat_models import ChatOpenAI
from langchain.callbacks import get_openai_callback

chatopenai = ChatOpenAI(model_name="gpt-3.5-turbo")

with get_openai_callback() as cb:
    query = "What is the distance to the Moon?"
    response = chatopenai.predict(query)
    print(response)

    query = "Who is the White Rabbit?"
    response = chatopenai.predict(query)
    print(response)

    print("\n")
    print(cb)


The average distance from the Earth to the Moon is about 238,900 miles (384,400 kilometers).
The White Rabbit is a fictional character from Lewis Carroll's novel "Alice's Adventures in Wonderland." He is a rabbit who wears a waistcoat and carries a pocket watch, always in a hurry and fearing that he is late. The White Rabbit is the character who leads Alice into Wonderland by running ahead of her and prompting her to follow him down the rabbit hole.


Tokens Used: 122
	Prompt Tokens: 28
	Completion Tokens: 94
Successful Requests: 2
Total Cost (USD): $0.00023
CPU times: user 11.1 ms, sys: 0 ns, total: 11.1 ms
Wall time: 2.5 s


## completion model

In [24]:
%%time
from langchain.llms import OpenAI

#llm = OpenAI(model_name="text-davinci-002", n=2, best_of=2)
llm = OpenAI(temperature=0.7)

with get_openai_callback() as cb:
    query = "What is the distance to the Moon?"
    response = llm.predict(query)
    print(response)

    query = "Who is the White Rabbit?"
    response = llm.predict(query)
    print(response)

    print("\n")
    print(cb)




The average distance from Earth to the Moon is 238,855 miles (384,400 kilometers).


The White Rabbit is a character from the novel Alice's Adventures in Wonderland and the Disney movie based on it. He is a talking rabbit who wears a waistcoat and carries a pocket watch. He is always late and talks to himself as he rushes about. He is often seen as a symbol of being late and procrastination.


Tokens Used: 103
	Prompt Tokens: 14
	Completion Tokens: 89
Successful Requests: 2
Total Cost (USD): $0.00206
CPU times: user 10.5 ms, sys: 0 ns, total: 10.5 ms
Wall time: 2.35 s


## cache

In [28]:
%%time
import langchain
from langchain.callbacks import get_openai_callback
from langchain.chat_models import ChatOpenAI
from langchain.cache import InMemoryCache

# cancel cache
langchain.llm_cache = None

chatopenai = ChatOpenAI(model_name="gpt-3.5-turbo")

with get_openai_callback() as cb:
    langchain.llm_cache = InMemoryCache()
    for _ in range(3):
        query = "What is the distance to the Moon?"
        response = chatopenai.predict(query)
        print(response)
        
        query = "Who is the White Rabbit?"
        response = chatopenai.predict(query)
        print(response)
    
        print("\n")
        print(cb)


The average distance from the Earth to the Moon is approximately 238,900 miles (384,400 kilometers).
The White Rabbit is a fictional character from Lewis Carroll's novel "Alice's Adventures in Wonderland." He is a talking rabbit dressed in a waistcoat and carrying a pocket watch, and he is known for being constantly late and in a hurry. The White Rabbit is the one who leads Alice into Wonderland by mistake, and she follows him down the rabbit hole.


Tokens Used: 121
	Prompt Tokens: 28
	Completion Tokens: 93
Successful Requests: 2
Total Cost (USD): $0.00022799999999999996
The average distance from the Earth to the Moon is approximately 238,900 miles (384,400 kilometers).
The White Rabbit is a fictional character from Lewis Carroll's novel "Alice's Adventures in Wonderland." He is a talking rabbit dressed in a waistcoat and carrying a pocket watch, and he is known for being constantly late and in a hurry. The White Rabbit is the one who leads Alice into Wonderland by mistake, and she fo

## benchmark

In [39]:
import langchain
from langchain.chat_models import ChatOpenAI
from langchain.callbacks import get_openai_callback
from langchain.llms import OpenAI
from datetime import datetime
from pprint import pprint

# cancel cache
langchain.llm_cache = None

#query = "What is the distance to the Moon?"
query = "Who is the White Rabbit?"

In [ ]:
pd.options.display.max_colwidth = 512
df.style.set_properties(**{'text-align': 'left'})

In [57]:
    
runs1 = []

def run_model(name, model):
    with get_openai_callback() as cb:
        tstart = datetime.now()
        response = model.predict(query)
        tend = datetime.now()
        elapsed =  tend - tstart
        nr_tokens_used = cb.total_tokens 
        total_cost = cb.total_cost
        metrics = {'name': name,
                   'response': response, 
                   'us': elapsed.microseconds, 
                   'cost': total_cost,
                   'tokens': nr_tokens_used}
        return metrics

chatopenai = ChatOpenAI(model_name="gpt-3.5-turbo")
runs1.append(run_model('gpt-3.5-turbo', chatopenai))

llm = OpenAI(temperature=0.7)
runs1.append(run_model('text-davinci-003_temp_0.7', llm))

#pprint(runs1)
df = pd.DataFrame(runs1)
df

,name,response,us,cost,tokens
0,gpt-3.5-turbo,"The White Rabbit is a character from Lewis Carroll's novel ""Alice's Adventures in Wonderland."" He is a talking rabbit who is always in a hurry and serves as Alice's guide throughout her adventures in Wonderland. The White Rabbit is known for wearing a waistcoat and carrying a pocket watch, constantly worrying about being late.",768477,0.000146,76
1,text-davinci-003_temp_0.7,"\n\nThe White Rabbit is a character from Lewis Carroll's Alice's Adventures in Wonderland. He is a white rabbit that Alice encounters who is hurrying along, muttering to himself, ""Oh dear! Oh dear! I shall be too late!"" He is late for an unspecified appointment and is seen carrying a pocket watch.",402988,0.001420,71


models https://platform.openai.com/docs/models/whisper
```
davinci	Most capable GPT-3 model. Can do any task the other models can do, often with higher quality.	2,049 tokens	Up to Oct 2019
curie	Very capable, but faster and lower cost than Davinci.	2,049 tokens	Up to Oct 2019
babbage	Capable of straightforward tasks, very fast, and lower cost.	2,049 tokens	Up to Oct 2019
ada	Capable of very simple tasks, usually the fastest model in the GPT-3 series, and lowest cost.	2,049 tokens	Up to Oct 2019
```

In [59]:
runs2 = []

llm = OpenAI(model_name="text-ada-001", temperature=0.7)
runs2.append(run_model('text-ada-001_temp_0.7', llm))

llm = OpenAI(model_name="text-babbage-001", temperature=0.7)
runs2.append(run_model('text-babbage-001_temp_0.7', llm))

llm = OpenAI(model_name="text-curie-001", temperature=0.7)
runs2.append(run_model('text-curie-001_temp_0.7', llm))

llm = OpenAI(model_name="text-davinci-002", temperature=0.7)
runs2.append(run_model('text-davinci-002_temp_0.7', llm))

#pprint(runs2)
df = pd.DataFrame(runs2)
df

,name,response,us,cost,tokens
0,text-ada-001_temp_0.7,"\n\nThe White Rabbit is a fictional character in Lewis Carroll's ""The Hunting of the White Rabbit"" story. Lewis Carroll is said to have based the character on his own WhiteKnight Chronicles persona.",507970,0.000018,46
1,text-babbage-001_temp_0.7,\n\nThe White Rabbit is a character in Lewis Carroll's novel Alice in Wonderland.,494185,0.000012,23
2,text-curie-001_temp_0.7,"\n\nThe White Rabbit is a character from Lewis Carroll's 1871 book, Alice's Adventures in Wonderland. He is a small, white rabbit who appears frequently in the book, often providing comic relief. He is also known for his cryptic advice to Alice.",740110,0.000116,58
3,text-davinci-002_temp_0.7,\n\nThe White Rabbit is a character from the 1865 novel Alice's Adventures in Wonderland by Lewis Carroll.,452411,0.000540,27


## merge

In [61]:
df.style.set_properties(**{'text-align': 'left'})

,name,response,us,cost,tokens
0,gpt-3.5-turbo,"The White Rabbit is a character from Lewis Carroll's novel ""Alice's Adventures in Wonderland."" He is a talking rabbit who is always in a hurry and serves as Alice's guide throughout her adventures in Wonderland. The White Rabbit is known for wearing a waistcoat and carrying a pocket watch, constantly worrying about being late.",768477,0.000146,76
1,text-davinci-003_temp_0.7,"The White Rabbit is a character from Lewis Carroll's Alice's Adventures in Wonderland. He is a white rabbit that Alice encounters who is hurrying along, muttering to himself, ""Oh dear! Oh dear! I shall be too late!"" He is late for an unspecified appointment and is seen carrying a pocket watch.",402988,0.001420,71
2,text-ada-001_temp_0.7,"The White Rabbit is a fictional character in Lewis Carroll's ""The Hunting of the White Rabbit"" story. Lewis Carroll is said to have based the character on his own WhiteKnight Chronicles persona.",507970,0.000018,46
3,text-babbage-001_temp_0.7,The White Rabbit is a character in Lewis Carroll's novel Alice in Wonderland.,494185,0.000012,23
4,text-curie-001_temp_0.7,"The White Rabbit is a character from Lewis Carroll's 1871 book, Alice's Adventures in Wonderland. He is a small, white rabbit who appears frequently in the book, often providing comic relief. He is also known for his cryptic advice to Alice.",740110,0.000116,58
5,text-davinci-002_temp_0.7,The White Rabbit is a character from the 1865 novel Alice's Adventures in Wonderland by Lewis Carroll.,452411,0.000540,27


In [62]:
#pprint(runs2)
df = pd.DataFrame(runs1 + runs2)
df

,name,response,us,cost,tokens
0,gpt-3.5-turbo,"The White Rabbit is a character from Lewis Carroll's novel ""Alice's Adventures in Wonderland."" He is a talking rabbit who is always in a hurry and serves as Alice's guide throughout her adventures in Wonderland. The White Rabbit is known for wearing a waistcoat and carrying a pocket watch, constantly worrying about being late.",768477,0.000146,76
1,text-davinci-003_temp_0.7,"\n\nThe White Rabbit is a character from Lewis Carroll's Alice's Adventures in Wonderland. He is a white rabbit that Alice encounters who is hurrying along, muttering to himself, ""Oh dear! Oh dear! I shall be too late!"" He is late for an unspecified appointment and is seen carrying a pocket watch.",402988,0.001420,71
2,text-ada-001_temp_0.7,"\n\nThe White Rabbit is a fictional character in Lewis Carroll's ""The Hunting of the White Rabbit"" story. Lewis Carroll is said to have based the character on his own WhiteKnight Chronicles persona.",507970,0.000018,46
3,text-babbage-001_temp_0.7,\n\nThe White Rabbit is a character in Lewis Carroll's novel Alice in Wonderland.,494185,0.000012,23
4,text-curie-001_temp_0.7,"\n\nThe White Rabbit is a character from Lewis Carroll's 1871 book, Alice's Adventures in Wonderland. He is a small, white rabbit who appears frequently in the book, often providing comic relief. He is also known for his cryptic advice to Alice.",740110,0.000116,58
5,text-davinci-002_temp_0.7,\n\nThe White Rabbit is a character from the 1865 novel Alice's Adventures in Wonderland by Lewis Carroll.,452411,0.000540,27


ATODO limiter la réponse
TODO instructions etre bref

In [55]:
pd.options.display.max_colwidth = 512

## extra - using chain

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

chatopenai = ChatOpenAI(model_name="gpt-3.5-turbo")

prompt = PromptTemplate (
    input_variables=["interest"],
    template="what are the 5 best countries in Europe ranked on {interest}"
)

llmchain_chat = LLMChain(llm=chatopenai, prompt=prompt)
print(llmchain_chat.run("food"))


In [12]:
import langchain
from langchain.llms import OpenAI

# To make the caching really obvious, lets use a slower model.
llm = OpenAI(model_name="text-davinci-002", n=2, best_of=2)